<a href="https://colab.research.google.com/github/Carlos-V-V/Gen-AI-for-CCM-pipeline-prototypes/blob/main/Simplified_pipeline_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DESCRIPTION OF THIS VERSION**

This notebook was created to remove the OT, HJB regularizers and just do the empirical risk reduction problem, i.e. train the neural networks by just minimizing the Wasserstein-2 distance between the predicted and observed point clouds (in 'path' formulation), to keep a simplified, first proof-of-concept version of the pipeline in case we need to go back to the original, simpler version.

This run of the pipeline assumes the system of equations:

dx_i/dt = Morse interactions

dp_i/dt = 0

(Active migration term has to be learned by the NN)

In [ ]:
!pip install torchdiffeq
!pip install geomloss

  Preparing metadata (setup.py) ... done
  Created wheel for geomloss: filename=geomloss-0.2.6-py3-none-any.whl size=32247 sha256=9752918ecfc5e728b6c5cbdf542e17d1b043253f2ff170655f93ce6ebbfda958
  Stored in directory: /root/.cache/pip/wheels/8c/4a/93/91d962ed04d2358b07000fb21b3164fd167b1b9cfddfce67fd
Successfully built geomloss


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchdiffeq import odeint  # from the `torchdiffeq' package
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cpu


We define the known kernels to use in this iteration. We define them like this so we can integrate the parameters into the optimizing pipeline, so that backpropagation flows through both the NN parameters (\phi) and the known kernel parameters (\theta). That is, the following is a trainable kernel model.  

For this example I will use only Morse potentials for dx/dt and a simple polarity term for dp/dt:

In [ ]:
class PosKernels(nn.Module):
  def __init__(self):
    super().__init__()
    self.A = nn.Parameter(torch.tensor(0.1))
    self.a = nn.Parameter(torch.tensor(1.0))
    self.R = nn.Parameter(torch.tensor(0.1))
    self.r = nn.Parameter(torch.tensor(0.5))
    #self.V = nn.Parameter(torch.tensor(1.0))


  def forward(self, x_ij, p_i):
    d = torch.norm(x_ij)
    epsilon = 1e-8  # Small epsilon to prevent division by zero
    return (-self.A*torch.exp(-d / (self.a + epsilon)) + self.R*torch.exp(-d / (self.r + epsilon)))*(x_ij / (d + epsilon)) #+ V*p_i  # Define assumed kernels

In [ ]:
class PolarityKernel(nn.Module):
    def __init__(self):
        super().__init__()
        self.g = nn.Parameter(torch.tensor(1.0)) # g is the parameter for the polarity equation, I'm just using one parameter for now

    def forward(self, x_ij, p_i, p_j):
        return 0  # Assumed polarity kernels

Now we define the NN for the value function U(t,x), which following the MFG formalism will also play the role of the learned interactions via **v_NN = -grad(U)**

In [ ]:
class ValueNet(nn.Module):
  def __init__(self, input_dim, output_dim, hidden_dim):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(input_dim, hidden_dim * 2),
        nn.Tanh(), # Or nn.ReLU()
        nn.Linear(hidden_dim * 2, hidden_dim * 2),
        nn.Tanh(), # Or nn.ReLU()
        nn.Linear(hidden_dim * 2, hidden_dim * 2),
        nn.Tanh(), # Or nn.ReLU()
        nn.Linear(hidden_dim * 2, output_dim)
        )

  def forward(self, t, x):
      if t.ndim == 1:
          t = t[:, None]
      return self.net(torch.cat([t, x], dim=-1))

Now we define the right-hand side of the ODEs in Eq (6) using both the ValueNet model above and the known kernels we defined.

NOTE that the following code **does not include polarity dynamics**:

In [ ]:
class ODESystem(nn.Module): # Defines ODE system as a PyTorch module
  def __init__(self, known_kernel_x, value_net, known_kernel_p):
    super().__init__()
    self.known_kernel_x = known_kernel_x
    self.value_net = value_net
    self.known_kernel_p = known_kernel_p

  def forward(self, t, X):  # Defines the ODE rhs dX/dt = f(t,X), where X = (x1,...xN,p1,...,pN) is the current state of the system
    N = X.shape[0]//2 # Define number of cells here! It's the shape of X divided by 2, since X has both positions and polarities.
    positions = X[:N] # Extracts positions, each entry is a 2D position vector
    polarities = X[N:]

    positions = positions.requires_grad_(True)

    '''---------------------------------------------------------
    THE NEXT PART IS THE KNOWN VELOCITY PART (pre-existing code)
    ------------------------------------------------------------
    '''

    # Vectorized calculation of pairwise differences and distances
    x_i = positions.unsqueeze(1) # Shape: (N, 1, D)
    x_j = positions.unsqueeze(0) # Shape: (1, N, D)
    x_ij = x_i - x_j            # Shape: (N, N, D)
    d_ij = torch.norm(x_ij, dim=-1) # Shape: (N, N)

    epsilon = 1e-8

    A = self.known_kernel_x.A                       # Modify here depending on parameters!
    a = self.known_kernel_x.a
    R = self.known_kernel_x.R
    r = self.known_kernel_x.r

    scalar_part = -A * torch.exp(-d_ij / a) + R * torch.exp(-d_ij / r)    # ***** MODIFY KERNELS HERE !!!

    scalar_part.diagonal(0).fill_(0.0)  # Avoids unphysical self-interactions

    d_ij_unit_vec = x_ij / (d_ij.unsqueeze(-1) + epsilon) # Normalizing
    d_ij_unit_vec[torch.eye(N, dtype=torch.bool, device=d_ij.device)] = 0.0 # Ensures diagonal vectors are 0, so there's no self-interactions

    known_x_forces_all_pairs = scalar_part.unsqueeze(-1) * d_ij_unit_vec  # multiplies the unit vector to give it the direction
    v_known = known_x_forces_all_pairs.sum(dim=1) # Sums over j to get the total 'known' force on each particle i, and forms the vector of all of them
                                                  # Shape becomes (N, 2) for the 2-D forces

    '''----------------------------------------------
    NOW COMES THE LEARNED PART:   v^NN = - grad(U)
    We'll feed ValueNet the per-particle state [x_i, p_i]
    -------------------------------------------------
    '''

    x_in = torch.cat([positions, polarities], dim=-1) # shape (N, 4) for (positions , polarities)

    t_batch = t.expand(N)   # This makes a time vector so ValueNet can be called "per particle" in batch
                            # This is because ValueNet expects a batch of vectors "batched" along time
                            # Each entry is the same time t
    U = self.value_net(t_batch, x_in)   # Evaluates the ValueNet at the inputs

    grad_pos = torch.autograd.grad(outputs = U.sum(), inputs=positions, create_graph=True)[0] # Gradient of U wrt positions, shape (N,2)
                                                                                              # bc position is 2D, so this is actually a Jacobian

    v_theta = -grad_pos

    dX = v_known + v_theta

    '''----------------------------------------------------------
    For polarity dynamics, this is zero for this proof of concept
    -------------------------------------------------------------
    '''

    dP = torch.zeros_like(polarities)

    return torch.cat([dX, dP], dim=0) # Concatenates dX and dP


Now we define the **cost functional**:

In [ ]:
from geomloss import SamplesLoss

# We define the Wasserstein-2 divergence, in this case approximated by a Sinkhorn divergence (which is optimized and faster than computing the exact Wasserstein distance)
W2_dist = SamplesLoss(loss="sinkhorn", p=2, blur=0.05)

# Both inputs must be (N, d) tensors (point clouds)
# Tune lambda depending on how strong you want the regularization term to be

def cost_function_W2_OT_HJB(simulated_trajectory, target_snapshots, value_net):
  # simulated_trajectory: (steps, N, dim) - The full trajectory of simulated positions
  # target_snapshot: (N, dim) - The single target point cloud for comparison
  # N and dim are global variables (N=20, dim=2).

  total_w2_loss = 0.0
  num_sim_steps = simulated_trajectory.shape[0] # Number of time steps in the simulated trajectory

  # Wasserstein-2 distance between simulated trajectory and target snapshot
  # We compare each simulated snapshot with the single target snapshot, and add them all together ('path' formulation)
  for j in range(num_sim_steps):
    sim_j_positions = simulated_trajectory[j] # This is a (N, dim) tensor for the j-th time step
    target_snapshot = target_snapshots[j]
    current_w2 = W2_dist(sim_j_positions, target_snapshot) # Compare with the single target
    total_w2_loss += current_w2

  return total_w2_loss/steps

We define the initial conditions, and the target dataset. For this, I upload a CSV file from a simulated dataset.

In [ ]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

Saving (Run=2.1)_V=0.025_A=0.025_R=0.0375_a=0.25_r=0.125.csv to (Run=2.1)_V=0.025_A=0.025_R=0.0375_a=0.25_r=0.125.csv


In [ ]:
df = pd.read_csv('(Run=2.1)_V=0.025_A=0.025_R=0.0375_a=0.25_r=0.125.csv', header=None)

In [ ]:
# We extract the first row of the dataset, i.e. the initial positions and polarities
row0 = df.iloc[0].values  # shape (80,1)

# We split into positions and polarities:
x0_flat = row0[:40]  # first 40 entries
p0_flat = row0[40:]  # next 40 entries

x_init_np = x0_flat.reshape(20, 2)  # shape (20, 2)
p_init_np = p0_flat.reshape(20, 2)  # shape (20, 2)

# Convert them to PyTorch tensors:
x_init = torch.tensor(x_init_np, dtype=torch.float32)  # shape (20, 2)
p_init = torch.tensor(p_init_np, dtype=torch.float32)  # shape (20, 2)

Now we define the "target" dataset as a tensor with shape (timeframes, 2*N, dim) comprised of the 'observed' synthetic data, with all the positions and then all the polarities for each timeframe.

In [ ]:
data_np = df.values  # Convert DataFrame to a NumPy array
# Reshape the data: 200 timeframes, 40 entities (positions + polarities), 2 dimensions (x,y or p_x, p_y)
target_snapshots = torch.tensor(data_np, dtype=torch.float32).reshape(200, 40, 2)
print(target_snapshots.shape)

torch.Size([200, 40, 2])


Now we initialize everything to start training:

In [ ]:
from re import X

N = 20
T_final = 100 # Final time for the simulation, MY DATA HAS T_final = 1000 with 200 timeframes, with time_step = 5
steps = 200 # Number of time steps, Michael used 200 steps
num_epochs = 100  # Number of training EPOCHS, EVENTUALLY DO 100-1000

# Dimensions for the NN models:
input_dim = 3 # This is the input dimension for *each pairwise interaction* (r, p_i)
hidden_dim = 64 # Lower to 32 if it takes too long
output_dim = 2 # Output dimension is 2 for a 2D force

value_input_dim = 1 + 4  # U input is [t] + [x,p], where x, p are 2D each
value_net = ValueNet(input_dim = value_input_dim, output_dim = 1, hidden_dim = hidden_dim)

# Initialize known position dynamics
known_x = PosKernels()

# And polarity dynamics
known_p = PolarityKernel()

# ODE system
ode_func = ODESystem(known_kernel_x = known_x, value_net = value_net, known_kernel_p = known_p)

# Define parameter groups with different learning rates

LR_NN = 1e-3
LR_known = 1e-3

param_groups = [
    {'params': value_net.parameters(), 'lr': LR_NN},
    {'params': known_x.parameters(), 'lr': LR_known},
    {'params': known_p.parameters(), 'lr': LR_known}
]


# Initial condition:
X0 = torch.cat([x_init, p_init], dim=0) # Change to match initial conditions in the dataset, and figure out why "dim=0"

# Time points:
t = torch.linspace(0, T_final, steps)

# Optimizer
optimizer = optim.AdamW(param_groups)

In [ ]:
# To run the pipeline on the GPU / CPU:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# These MUST be moved:
X0 = X0.to(device)
target_snapshots = target_snapshots.to(device)
t = t.to(device)
value_net = value_net.to(device)
ode_func = ode_func.to(device)


We initialize the weights and biases of the neural networks to smaller values. This helps control the initial magnitude of the unknown forces

In [ ]:
def initialize_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight, gain=0.1) # Using Xavier uniform with a small gain
        if m.bias is not None:
            nn.init.constant_(m.bias, 0) # Initialize biases to zero

value_net.apply(initialize_weights)

print("Neural network weights initialized.")

Neural network weights initialized.


We extract the initial kernel parameters (before training):

In [ ]:
for name, param in known_x.named_parameters():
    print(f"{name}: {param.data}")

A: 0.10277709364891052
a: 1.0033169984817505
R: 0.09749865531921387
r: 0.49702003598213196


The next cell tests how long it takes to integrate the NN-augmented ODE system. Here we re-define the ValueNet neural network because it was not recognizing it before.

In [ ]:
import time
import torch.nn as nn
start = time.time()

# Re-define ValueNet class here to ensure it is correctly loaded
class ValueNet(nn.Module):
  def __init__(self, input_dim, output_dim, hidden_dim):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.Tanh(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.Tanh(),
        nn.Linear(hidden_dim, output_dim)
        )

  def forward(self, t, x):
      # t: (B,1) or (B,), x: (B,d)
      if t.ndim == 1:
          t = t[:, None]
      return self.net(torch.cat([t, x], dim=-1))  # (B,1)

# Re-instantiate value_net and ode_func to ensure correct forward method is recognized
# The variables (value_input_dim, output_dim, hidden_dim, known_x, known_p, device) are defined in previous cells.

value_net = ValueNet(input_dim = value_input_dim, output_dim = 1, hidden_dim = hidden_dim)
ode_func = ODESystem(known_kernel_x = known_x, value_net = value_net, known_kernel_p = known_p)

# Apply weight initialization and move to device again for the new instances
value_net.apply(initialize_weights)
value_net = value_net.to(device)

ode_func = ode_func.to(device)

X_pred = odeint(ode_func, X0, t, method='dopri5', rtol=1e-3, atol=1e-4)

print("Integration time:", time.time() - start)


Integration time: 0.657879114151001


Below is a **protoype** of the new training loop. Use the following cell to **test changes only**.

In [ ]:
# Test to see how long it takes to run one epoch of training:

import time
start = time.time()

dim = 2 # Dimension of position & polarity vectors

optimizer.zero_grad()

# Assuming X_pred is (steps, 2*N, D) and t is (steps,)

X_pred = odeint(ode_func, X0, t, method='dopri5', rtol=1e-3, atol=1e-4)
Pred_pos = X_pred[:, :N] # Corrected slicing for positions over all time steps
Pred_pol = X_pred[:, N:] # Corrected slicing for polarities over all time steps

loss = cost_function_W2_OT_HJB(X_pred, target_snapshots, value_net)
loss.backward()
optimizer.step()

print("One epoch of training:", time.time() - start , f"Loss: {loss.item()}")


One epoch of training: 2.471339464187622 Loss: 9464.6484375


Now we do the **full** training, training for a total of "num_epochs" epochs (per training round).

In [ ]:
num_epochs = 100

param_groups = [
    {'params': value_net.parameters(), 'lr': LR_NN},
    {'params': known_x.parameters(), 'lr': LR_known}
] # Add weight decay for more stable and efficient training

optimizer = optim.AdamW(param_groups)

LR_NN = 1e-3
LR_known = 1e-3

rtol = 1e-3
atol = 1e-4

# Automate the training loop for num_epochs times
import time

total_training_start_time = time.time()

# num_epochs, N, dim, alpha1, alpha2, optimizer, ode_func, X0, t,
# value_net, Hamiltonian, Target_pos are assumed to be defined globally

print(f"Starting training for {num_epochs} epochs...")
for epoch in range(num_epochs):
  epoch_start_time = time.time() # Start timer for this epoch

  optimizer.zero_grad()
  X_pred = odeint(ode_func, X0, t, method='dopri5', rtol=rtol, atol=atol)
  Pred_pos = X_pred[:, :N]
  Pred_pol = X_pred[:, N:]

  # Calculate the loss (a.k.a. cost)
  loss = cost_function_W2_OT_HJB(X_pred, target_snapshots, value_net)
  loss.backward()
  optimizer.step()

  print(f"Epoch {epoch+1}/{num_epochs} | Time: {time.time() - epoch_start_time:.4f}s | Loss: {loss.item():.4f}")

total_training_end_time = time.time()
print(f"Total training for {num_epochs} epochs completed in {total_training_end_time - total_training_start_time:.4f}s")

Extract the final optimized parameters (after all training rounds) from the known kernels NN:

In [ ]:
# We extract the final optimized parameters from the known_x NN:
optimized_known_x_params = known_x.parameters()

print("Optimized parameters:")
param_data = {}
for name, param in known_x.named_parameters():
    if param.requires_grad:
        print(name, param.data)
        param_data[name] = param.data.item() # Store parameter name and value

# Save parameters to a txt file
with open("Learned_kernel_params.txt", "w") as f:
    for name, value in param_data.items():
        f.write(f"{name}: {value}\n")

print("Optimized parameters saved to Learned_kernel_params.txt")

Optimized parameters:
A tensor(0.0611)
a tensor(0.1982)
R tensor(0.0809)
r tensor(0.1279)
Optimized parameters saved to Learned_kernel_params.txt


In [ ]:
files.download("Learned_kernel_params.txt")

We save the trained NN models for later use:

In [ ]:
# Save the trained NN models
torch.save(value_net.state_dict(), 'value_net_model.pth')
torch.save(known_x.state_dict(), 'known_x_model.pth')
torch.save(known_p.state_dict(), 'known_p_model.pth')

print("Trained NN models saved.")

Trained NN models saved.


In [ ]:
files.download('value_net_model.pth')
files.download('known_x_model.pth')
files.download('known_p_model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>